# Week 8: Vector DB / Retrieval

SiteLens AI — Inspection-precedent retrieval prototype.

**Topic:** NLP, vector databases, RAG  
**Dates:** May 10–14 2026  
**Deliverable:** Pinecone + sentence-transformers retrieval prototype for inspection precedents.

**Status:** In progress.

---

| Cell | Type | Purpose |
|---|---|---|
| header | md | This cell |
| 1 | code | SETUP — install dependencies (run once, then comment out) |
| 2 | code | SETUP — imports, env, Pinecone client |
| 3 | code | DATA — text helpers (damage labels, MMI scale, row_to_text) |
| 4 | code | DATA — load sample records from committed JSON |
| 5 | code | EMBED — encode records with sentence-transformers |
| 6 | code | INDEX — create Pinecone index (drop-if-exists, wait for ready) |
| 7 | code | UPSERT — push vectors to Pinecone |
| 8 | code | QUERY — retrieve top-3 nearest neighbours |

In [ ]:
# 1 SETUP — install dependencies (run once, then comment out)
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass  # Jupyter wraps stdout; reconfigure not available
# %pip install pinecone sentence-transformers python-dotenv

In [ ]:
#2 SETUP — imports, env, Pinecone client
import os
import pandas as pd
import geopandas as gpd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec

load_dotenv("../.env")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME", "sitelens")

assert PINECONE_API_KEY, "PINECONE_API_KEY not loaded — check .env path"
pc = Pinecone(api_key=PINECONE_API_KEY)
print("Key loaded:", PINECONE_API_KEY[:8], "...")

In [ ]:
#3 DATA — text helpers (damage labels, MMI scale, row_to_text)

MMI_LABELS = [(9, "violent"), (8, "severe"), (7, "very strong"),
              (6, "strong"), (4, "moderate"), (0, "light")]

DAMAGE_TEXT = {0: "survived", 1: "destroyed", 9: "obstructed", 99: "inconsistent footprint"}

def mmi_label(mmi):
    for threshold, label in MMI_LABELS:
        if mmi >= threshold:
            return label
    return "light"

def row_to_text(row):
    outcome = DAMAGE_TEXT.get(row["damage_val"], "unknown")
    hazards = [h for h, flag in [("fire", row.get("GSI_fire")),
               ("tsunami", row.get("GSI_tsunami")),
               ("slope failure", row.get("GSI_slope_failure"))] if flag == 1]
    haz_str = ", ".join(hazards) if hazards else "seismic only"
    mmi_str = f"MMI {row.get('USGS_MMI', 0):.1f} ({mmi_label(row.get('USGS_MMI', 0))} shaking)"
    return f"Building {outcome}. Hazard: {haz_str}. {mmi_str}."


In [ ]:
# 4 DATA — load sample records from committed JSON
import json

with open("../data/samples/sample_records.json", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records)} records")
for r in records[:2]:
    print(r["text"])

In [ ]:
#5 EMBED — encode records with sentence-transformers

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([r["text"] for r in records], show_progress_bar=True)
print(f"Shape: {embeddings.shape}")


In [ ]:
#6 INDEX — create Pinecone index (drop-if-exists, wait for ready)

if INDEX_NAME in pc.list_indexes().names():
    pc.delete_index(INDEX_NAME)

pc.create_index(
    name=INDEX_NAME,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)

import time
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

index = pc.Index(INDEX_NAME)
print(index.describe_index_stats())


In [ ]:
# 7 UPSERT — push vectors to Pinecone
vectors = [
    {"id": records[i]["id"],
     "values": embeddings[i].tolist(),
     "metadata": {"text": records[i]["text"], "municipality": records[i].get("municipality", "")}}
    for i in range(len(records))
]
index.upsert(vectors=vectors)
print(f"Upserted {len(vectors)} vectors")

In [ ]:
#8 QUERY — retrieve top-3 nearest neighbours

import time
time.sleep(5)

queries = [
    "building destroyed by fire in dense urban area",
    "structure survived tsunami zone with strong shaking",
]

for query in queries:
    vec = model.encode([query])[0].tolist()
    results = index.query(vector=vec, top_k=3, include_metadata=True)
    print(f"\nQuery: '{query}'")
    for m in results["matches"]:
        print(f"  [{m['score']:.3f}] {m['id']}  {m['metadata']['text']}")

